# 🏥 General Health Query Chatbot
## Task 4 – AI/ML Internship Project

---

### 🎯 Problem Statement
People often have general health questions but don't always have immediate access to medical professionals. This chatbot provides **friendly, safe, and clear** answers to general health queries using a Large Language Model (LLM).

### 🏁 Objective
- Build a conversational health assistant using **Google Gemini API** (free tier)
- Use **prompt engineering** to make responses friendly, clear, and responsible
- Implement **safety filters** to avoid harmful medical advice
- Handle example queries like *"What causes a sore throat?"* or *"Is paracetamol safe for children?"*

### 🛠️ Tools Used
- **Google Gemini 1.5 Flash** (free LLM API)
- **Prompt Engineering** for persona and safety
- **Python** for scripting the chatbot logic

---
## 📚 Section 1: Install & Import Libraries

In [ ]:
# Install the Google Generative AI library
!pip install google-generativeai -q
print('✅ Libraries installed!')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import google.generativeai as genai
import textwrap
import re

print('✅ All libraries imported successfully!')

---
## 🔑 Section 2: Configure API Key

Get your **free API key** from 👉 [aistudio.google.com](https://aistudio.google.com)

Steps:
1. Sign in with your Google account
2. Click **"Get API Key"**
3. Click **"Create API Key"**
4. Copy and paste it below

In [ ]:
# ── Paste your free Gemini API key here ───────────────────────────────────────
API_KEY = "YOUR_GEMINI_API_KEY_HERE"   # 🔑 Replace with your actual key

# Configure the Gemini API
genai.configure(api_key=API_KEY)

# Use Gemini 1.5 Flash — fast and completely free
model = genai.GenerativeModel('gemini-1.5-flash')

print('✅ Gemini API configured successfully!')

---
## 🧠 Section 3: Prompt Engineering

Prompt engineering is the practice of **designing instructions** that guide the LLM to behave in a specific way.

Our system prompt does the following:
- Gives the AI a **friendly medical assistant persona**
- Instructs it to use **simple, clear language**
- Enforces **safety rules** — always recommend seeing a doctor for serious issues
- Prevents the AI from giving **specific dosage or prescription advice**

In [ ]:
# ── System Prompt (Prompt Engineering) ───────────────────────────────────────
SYSTEM_PROMPT = """
You are HealthBot, a friendly and knowledgeable general health assistant.

Your role:
- Answer general health questions in a warm, clear, and easy-to-understand way
- Use simple language that anyone can understand (avoid complex medical jargon)
- Keep responses concise but informative (3-5 sentences ideally)
- Always show empathy and care in your responses

Safety rules you MUST follow:
- NEVER provide specific medication dosages or prescriptions
- ALWAYS recommend consulting a real doctor for serious, severe, or persistent symptoms
- NEVER diagnose a specific disease or condition
- If a question sounds like a medical emergency, immediately tell the user to call emergency services
- Add a gentle reminder at the end that you are an AI and not a substitute for professional medical advice

Tone: Friendly, caring, calm, and professional — like a knowledgeable friend who happens to know a lot about health.
"""

print('✅ System prompt defined!')
print('\n📋 Prompt Preview:')
print(SYSTEM_PROMPT)

---
## 🛡️ Section 4: Safety Filters

We add an extra layer of **keyword-based safety filtering** on top of the LLM's built-in safety:

- Detects **emergency keywords** and immediately redirects to emergency services
- Detects **harmful/dangerous queries** and refuses to answer
- Acts as a **pre-filter** before the query even reaches the LLM

In [ ]:
# ── Safety Filter Keywords ────────────────────────────────────────────────────

# Emergency situations — redirect to emergency services immediately
EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', 'can\'t breathe', 'cannot breathe',
    'difficulty breathing', 'stroke', 'unconscious', 'not breathing',
    'severe bleeding', 'overdose', 'suicide', 'poisoning', 'seizure'
]

# Harmful queries — refuse to answer
HARMFUL_KEYWORDS = [
    'how to overdose', 'lethal dose', 'kill myself', 'how much to take to die',
    'how to get high on', 'recreational drug'
]


def safety_filter(query: str) -> tuple[bool, str]:
    """
    Pre-checks the user query for emergencies or harmful intent.
    Returns (is_safe, message):
      - If emergency: (False, emergency_message)
      - If harmful:   (False, refusal_message)
      - If safe:      (True, "")
    """
    query_lower = query.lower()

    # Check for emergency keywords
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return False, (
                "🚨 This sounds like a medical emergency!\n"
                "Please call emergency services immediately:\n"
                "  🇵🇰 Pakistan  : 115 (Rescue) or 1122\n"
                "  🌍 Universal : 112\n\n"
                "Do not wait — please seek help right now!"
            )

    # Check for harmful keywords
    for keyword in HARMFUL_KEYWORDS:
        if keyword in query_lower:
            return False, (
                "⚠️ I'm sorry, but I can't help with that request.\n"
                "If you're going through a difficult time, please reach out:\n"
                "  💚 Talk to someone you trust\n"
                "  📞 Contact a mental health helpline\n"
                "  🏥 Visit your nearest hospital"
            )

    return True, ""


print('✅ Safety filters defined!')

---
## 🤖 Section 5: Chatbot Core Function

This is the main function that:
1. Runs the query through **safety filters**
2. Builds the **full prompt** (system + conversation history + new query)
3. Sends it to the **Gemini API**
4. Returns the formatted response

In [ ]:
# ── Conversation History (maintains context across turns) ─────────────────────
conversation_history = []


def ask_healthbot(user_query: str, verbose: bool = True) -> str:
    """
    Main chatbot function.
    - Applies safety filters
    - Sends query to Gemini with system prompt + history
    - Returns the chatbot response
    """
    global conversation_history

    # ── Step 1: Safety Filter ─────────────────────────────────────────────────
    is_safe, safety_message = safety_filter(user_query)
    if not is_safe:
        if verbose:
            print(f"\n{'='*60}")
            print(f"👤 You: {user_query}")
            print(f"{'='*60}")
            print(f"🏥 HealthBot:\n{safety_message}")
            print(f"{'='*60}\n")
        return safety_message

    # ── Step 2: Build Full Prompt ─────────────────────────────────────────────
    # Combine system prompt + conversation history + new query
    history_text = ""
    for turn in conversation_history[-6:]:   # keep last 3 turns for context
        history_text += f"User: {turn['user']}\nHealthBot: {turn['bot']}\n\n"

    full_prompt = f"{SYSTEM_PROMPT}\n\n{history_text}User: {user_query}\nHealthBot:"

    # ── Step 3: Call Gemini API ───────────────────────────────────────────────
    try:
        response = model.generate_content(full_prompt)
        bot_response = response.text.strip()
    except Exception as e:
        bot_response = f"Sorry, I encountered an error: {str(e)}. Please try again."

    # ── Step 4: Save to History ───────────────────────────────────────────────
    conversation_history.append({'user': user_query, 'bot': bot_response})

    # ── Step 5: Display ───────────────────────────────────────────────────────
    if verbose:
        print(f"\n{'='*60}")
        print(f"👤 You: {user_query}")
        print(f"{'='*60}")
        print(f"🏥 HealthBot:\n{bot_response}")
        print(f"{'='*60}\n")

    return bot_response


def reset_conversation():
    """Clear the conversation history to start fresh."""
    global conversation_history
    conversation_history = []
    print('🔄 Conversation history cleared!')


print('✅ Chatbot functions defined!')

---
## 🧪 Section 6: Test the Chatbot

Let's test with the example queries from the task instructions!

In [ ]:
# ── Test 1: Common health question ────────────────────────────────────────────
ask_healthbot("What causes a sore throat?")

In [ ]:
# ── Test 2: Medication safety question ───────────────────────────────────────
ask_healthbot("Is paracetamol safe for children?")

In [ ]:
# ── Test 3: General symptoms question ────────────────────────────────────────
ask_healthbot("I have been having headaches every morning, what could be the reason?")

In [ ]:
# ── Test 4: Diet and lifestyle question ──────────────────────────────────────
ask_healthbot("How much water should I drink every day?")

In [ ]:
# ── Test 5: Safety filter — Emergency detection ───────────────────────────────
ask_healthbot("I think I am having a heart attack, what should I do?")

In [ ]:
# ── Test 6: Multi-turn conversation (context awareness) ───────────────────────
reset_conversation()
ask_healthbot("I have a cold and my nose is blocked.")
ask_healthbot("What home remedies can help with this?")   # refers to the cold above

---
## 💬 Section 7: Interactive Chat Mode

Run this cell for a **live chat session** with HealthBot!
Type `quit` or `exit` to end the session, or `reset` to clear history.

In [ ]:
# ── Interactive Chat Loop ─────────────────────────────────────────────────────
print('🏥 Welcome to HealthBot!')
print('   Ask me any general health question.')
print('   Type "reset" to clear history, "quit" to exit.\n')

reset_conversation()

while True:
    user_input = input('👤 You: ').strip()

    if not user_input:
        continue

    if user_input.lower() in ['quit', 'exit', 'bye']:
        print('🏥 HealthBot: Take care and stay healthy! Goodbye! 👋')
        break

    if user_input.lower() == 'reset':
        reset_conversation()
        continue

    ask_healthbot(user_input)

---
## 📊 Section 8: Prompt Engineering Analysis

Let's compare responses **with and without** our system prompt to demonstrate the impact of prompt engineering.

In [ ]:
test_query = "What causes a fever?"

# ── Without Prompt Engineering ────────────────────────────────────────────────
raw_response = model.generate_content(test_query)
print('❌ WITHOUT Prompt Engineering:')
print('='*60)
print(raw_response.text.strip())
print()

# ── With Prompt Engineering ───────────────────────────────────────────────────
print('✅ WITH Prompt Engineering (HealthBot):')
print('='*60)
engineered_prompt = f"{SYSTEM_PROMPT}\n\nUser: {test_query}\nHealthBot:"
engineered_response = model.generate_content(engineered_prompt)
print(engineered_response.text.strip())

---
## 💡 Section 9: Results & Final Insights

### Key Findings

1. **Prompt Engineering works** — the system prompt successfully transformed a generic LLM into a focused, friendly, and responsible health assistant. Responses with the prompt were more empathetic, structured, and included safety disclaimers.

2. **Two-layer safety system** — combining keyword-based pre-filtering with LLM-level safety guardrails provides robust protection against harmful outputs:
   - **Layer 1:** Keyword filter catches emergencies and harmful queries instantly
   - **Layer 2:** System prompt instructs the LLM to always recommend professional help

3. **Context awareness** — the chatbot maintains conversation history, allowing it to understand follow-up questions (e.g., "What about that?" referring to a previous message).

4. **Free and accessible** — using Google Gemini's free tier makes this solution accessible without any cost, making it practical for real-world deployment.

### Limitations
- The chatbot provides **general information only** — it cannot replace professional medical diagnosis
- Keyword filters may **miss creative wordings** of harmful queries
- LLMs can occasionally **hallucinate** medical information — always verify with a doctor
- No **persistent memory** across sessions (history resets each run)

### Future Improvements
- Add a **Gradio or Streamlit UI** for a proper web interface
- Integrate a **medical knowledge base (RAG)** for more accurate responses
- Add **multi-language support** for regional languages
- Implement **conversation logging** for quality monitoring
- Add **symptom checker** with structured follow-up questions